In [1]:
import pandas as pd
import re
import os

## Delete all old files

In [2]:
# Delete all files under target_dir recursively, except keep_file
target_dir = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し"
keep_file = "CombineMitoshi.ipynb"

for root, dirs, files in os.walk(target_dir):
    for fname in files:
        if fname == keep_file:
            continue
        fpath = os.path.join(root, fname)
        try:
            os.remove(fpath)
            print(f"Deleted: {fpath}")
        except Exception as e:
            print(f"Failed: {fpath} -> {e}")

print("Done.")

Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result\イノチオみらい㈱.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result\本社収支.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result\生産課.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result\生産部.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result\青果流通課.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリ_組織_Result\イノチオアグリ㈱.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリ_組織_Result\イノチオアグリ本社.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリ_組織_Result\システム設計課.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリ_組織_Result\フィルム加工課 出荷部門.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\

## Separate each sheets (組織名) into a file

In [21]:
# Separate 組織 of each company

# === 0) 入出力フォルダを設定 ===
input_dir = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\アグリコ"
save_dir  = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result"

os.makedirs(save_dir, exist_ok=True)

def _norm(s):
    """列名の揺れに強くする簡易正規化（全角/半角・空白除去・小文字化）"""
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\u3000", "").replace(" ", "")
    s = s.replace("Ｍ", "M").replace("Ｐ", "P").replace("／", "/")
    return s.lower().strip()

def process_one_file(path, save_dir):
    try:
        raw = pd.read_csv(path, encoding="utf-8-sig", header=None)
        if raw.empty or len(raw) < 6:
            print(f"SKIP (empty or too few rows): {path}")
            return

        # --- 1) 組織情報抽出（2行目） ---
        row2 = raw.iloc[1].astype(str)
        first_non_empty = next((v for v in row2 if v and v.lower() != "nan"), "")

        org_line = (
            first_non_empty
            .replace("\u3000", "")  # 全角スペース除去
            .strip()
            .replace("＿", "_")     # 全角アンダースコア→半角
        )

        # "230H60200_総務課" のような形式を分割
        org_code, org_name = "", ""
        m = re.match(r"^\s*([0-9A-Za-z]+)_(.+?)\s*$", org_line)
        if m:
            org_code, org_name = m.group(1), m.group(2).strip()
        else:
            if "_" in org_line:
                parts = org_line.split("_", 1)
                org_code, org_name = parts[0].strip(), parts[1].strip()
            else:
                org_name = org_line.strip()

        # --- 2) ヘッダ構築 ---
        if len(raw) < 5:
            print(f"SKIP (header rows missing): {path}")
            return

        top = raw.iloc[3].astype(str).str.strip().replace({"nan": ""}).str.replace(" ", "", regex=False)
        sub = raw.iloc[4].astype(str).str.strip().replace({"nan": ""}).str.replace(" ", "", regex=False)

        filled_top, last = [], ""
        for val in top:
            if val:
                last = val
                filled_top.append(val)
            else:
                filled_top.append(last)

        multi_cols = pd.MultiIndex.from_tuples(list(zip(filled_top, sub)))
        data = raw.iloc[5:].reset_index(drop=True)
        # 列数一致チェック（ズレ対策）
        if data.shape[1] != len(multi_cols):
            min_cols = min(data.shape[1], len(multi_cols))
            data = data.iloc[:, :min_cols]
            multi_cols = multi_cols[:min_cols]
        data.columns = multi_cols

        # === 列探索ユーティリティ ===
        def find_subject_col(df_cols):
            # サブヘッダが「科目」
            subject_candidates = [c for c in df_cols if _norm(c[1]) == _norm("科目")]
            if subject_candidates:
                return subject_candidates[0]
            # 文字列に「科目」を含む
            subject_candidates = [c for c in df_cols if "科目" in "".join(map(str, c))]
            return subject_candidates[0] if subject_candidates else None

        def find_mitoshi_col(df_cols):
            """
            「見通し」を優先して探す。
            ・(top='年度', sub='見通し') を最優先
            ・それがなければ、サブヘッダが '見通し' の最初の列
            """
            # 厳密に年度_見通し
            strict = [c for c in df_cols if _norm(c[0]) == _norm("年度") and _norm(c[1]) == _norm("見通し")]
            if strict:
                return strict[0]
            # サブが見通しならOK（上段は何でも良い）
            loose = [c for c in df_cols if _norm(c[1]) == _norm("見通し")]
            return loose[0] if loose else None

        def find_under_same_top(df_cols, top_label, sub_name_candidates):
            """
            上段ヘッダが top_label（＝年度_見通しの上段）と一致し、
            サブヘッダが候補に一致する列を返す。
            """
            sub_keys = {_norm(s) for s in sub_name_candidates}
            for c in df_cols:
                if _norm(c[0]) == _norm(top_label) and _norm(c[1]) in sub_keys:
                    return c
            # %表記の揺れ（例：'MP比(%)'）を緩く拾う
            for c in df_cols:
                if _norm(c[0]) == _norm(top_label):
                    ns = _norm(c[1])
                    for k in list(sub_keys):
                        if k and (k in ns):  # 'mp比' in 'mp比(%)' など
                            return c
            return None

        # 科目列
        subject_col = find_subject_col(data.columns)
        if subject_col is None:
            print(f"SKIP (科目列が見つからない): {path}")
            return

        # 年度_見通し列
        col_mitoshi = find_mitoshi_col(data.columns)
        if col_mitoshi is None:
            print(f"SKIP (年度_見通し列が見つからない): {path}")
            return

        # ★ 同じ「上段ヘッダ（＝年度_見通しの top）」から MP / MP比 / 前年比 を探す
        the_top = col_mitoshi[0]  # これが「その年度」（同じグループ）を指す

        # 候補（揺れ対応）
        mp_candidates = ["M/P", "MP", "Ｍ／Ｐ"]
        mp_ratio_candidates = ["M/P比", "M/P比(%)", "MP比", "MP比(%)"]
        yoy_candidates = ["前年比", "前年比(%)"]

        col_mp       = find_under_same_top(data.columns, the_top, mp_candidates)
        col_mp_ratio = find_under_same_top(data.columns, the_top, mp_ratio_candidates)
        col_yoy      = find_under_same_top(data.columns, the_top, yoy_candidates)

        # 出力を構成（年度_見通しと科目は必須、MP/MP比/前年比は見つかった分だけ）   
        use_cols = [subject_col, col_mitoshi]
        if col_mp is not None:       use_cols.append(col_mp)
        if col_mp_ratio is not None: use_cols.append(col_mp_ratio)
        if col_yoy is not None:      use_cols.append(col_yoy)

        out = data[use_cols].copy()

        # 列名をフラットに命名
        rename_map = {subject_col: "科目", col_mitoshi: "年度_見通し"}
        if col_mp is not None:       rename_map[col_mp] = "MP"
        if col_mp_ratio is not None: rename_map[col_mp_ratio] = "MP比"
        if col_yoy is not None:      rename_map[col_yoy] = "前年比"
        out.columns = [rename_map.get(c, "_".join(map(str, c))) for c in out.columns]

        # --- 3) クリーニング ---
        # 科目
        out["科目"] = (
            out["科目"].astype(str)
            .str.replace("\u3000", "", regex=False)
            .str.strip()
        )
        out = out[out["科目"].notna() & (out["科目"] != "") & (out["科目"].str.lower() != "nan")]

        # 数値列クリーニング
        def clean_numeric_series(s):
            return (
                s.astype(str)
                 .str.replace(",", "", regex=False)
                 .str.replace("¥", "", regex=False)
                 .str.replace("%", "", regex=False)
                 .str.replace("\u3000", "", regex=False)
                 .str.strip()
            )

        for col in ["年度_見通し", "MP", "MP比", "前年比"]:
            if col in out.columns:
                out[col] = clean_numeric_series(out[col])
                out[col] = pd.to_numeric(out[col], errors="coerce")

        # --- 4) 組織コード/名 ---
        org_code2, org_name2 = "", ""
        if "_" in org_line:
            org_code2, org_name2 = org_line.split("_", 1)
            org_code2 = org_code2.strip()
            org_name2 = org_name2.strip()
        else:
            org_name2 = org_line.strip()

        invalid_chars = r'\/:*?"<>|'
        org_name_sanitized = re.sub(f"[{re.escape(invalid_chars)}]", "", org_name2).strip()

        out["組織コード"] = org_code if org_code else org_code2
        out["組織名"]   = org_name if org_name else org_name2

        # --- 5) 必ず列を用意し、順序を固定 ---
        for need in ["年度_見通し", "MP", "MP比", "前年比"]:
            if need not in out.columns:
                out[need] = pd.NA
        out = out[["組織コード", "組織名", "科目", "年度_見通し", "MP", "MP比", "前年比"]]

        # --- 6) Save ---
        save_path = os.path.join(save_dir, f"{org_name_sanitized}.csv")
        out.to_csv(save_path, index=False, sep="\t", encoding="utf-8-sig")
        print(f"SAVED: {save_path}")

    except Exception as e:
        print(f"ERROR: {path} -> {e}")

# === ループ処理 ===
for fname in os.listdir(input_dir):
    if not fname.lower().endswith(".csv"):
        continue
    fpath = os.path.join(input_dir, fname)
    if os.path.isfile(fpath):
        process_one_file(fpath, save_dir)

SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\㈱アグリコ.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\本社収支.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\アグリコ.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\営業部.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\園芸センター.csv


## Combine files

In [22]:
#Combine 組織

import glob
import pandas as pd

# # Tìm tất cả các file .txt trong thư mục gốc và các thư mục con
file_list = glob.glob(f"{save_dir}\\*.csv")

# Các cột cần ép kiểu float
float_columns = ["年度_見通し", "MP", "MP比", "前年比"]

# Đọc và nối tất cả các file
dfs = []
for f in file_list:
    with open(f, 'r', encoding='utf-8-sig', errors='ignore') as file:
        if '<!DOCTYPE HTML' in file.read():
            print(f"Bỏ qua file HTML: {f}")
            continue

    df = pd.read_csv(f, encoding="utf-8-sig", delimiter='\t', on_bad_lines='skip', dtype={'組織コード': str})
    for col in float_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('-', 0), errors='coerce')
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df
# Hiển thị số lượng file đã đọc và số dòng tổng cộng
print(f"Đã đọc {len(file_list)} file.")
print(f"Tổng số dòng: {len(df)}")

Đã đọc 5 file.
Tổng số dòng: 270


In [23]:
df.to_csv("見通し_アグリコ.csv", index=False, sep='\t', encoding="utf-8-sig")